# NBA Points Modeling Research (v1)

Context:
- Goal: build a game-to-game points modeling workflow, graduating from exploratory notebook → pipeline scripts.
- Focus in this notebook: data sanity, target construction, feature diagnostics, baseline model, edge calibration.
- Scope: research only (no production assumptions yet).
- Season hard-coded to 2025-26 throughout; change `SEASON` below to explore others.

Data source: `s3://nba-betting-mt/data/03_intermediate/player_props_with_actuals_{season}.csv`
- One row per player-game (already points-only, aggregated across bookmakers)
- Key columns: `points_line`, `points_over_odds`, `points_under_odds`, `PTS`, `MIN`, `team_spread`, `scorer_type`, `is_home`

## TL;DR

**Result: No profitable edge found with current feature set.**

| | |
|---|---|
| **Data** | 3 seasons (2023-24 → 2025-26), ~36k player-game rows, odds-filtered (market P 0.35–0.65) |
| **Models tried** | OLS, Ridge, GBM (regression) · LogReg, XGBoost (classification) |
| **Best model** | LogReg (rolling pts + line) — beats market Brier by ~0.001–0.003, consistent across folds |
| **ROI** | Fold1 (test 2024-25): -5% to -7%. Fold2 (test 2025-26): ~flat. Not enough to beat 6.9% vig. |
| **Key finding** | Market line captures 97.97% of signal — rolling stats add almost nothing |
| **Best signal** | Rim attackers hit unders at 57% vs 51% overall — untested as standalone filter |

**Next:** `v2_points_rim_attacker_filter.ipynb` — test rim attacker under filter walk-forward.


In [ ]:
from pathlib import Path
import subprocess
import sys

import boto3
import numpy as np
import pandas as pd
from io import StringIO

repo_root = Path(
    subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip()
)
module_root = repo_root / "src" / "nba_points_modeling"
research_root = module_root / "research"

sys.path.insert(0, str(repo_root / "src"))

# Hard-coded season — change here to explore other seasons
SEASON = "2025-26"

S3_BUCKET_INTERMEDIATE = "nba-betting-mt"
S3_KEY_INTERMEDIATE = f"data/03_intermediate/player_props_with_actuals_{SEASON}.csv"

print(f"repo_root={repo_root}")
print(f"module_root={module_root}")
print(f"season={SEASON}")

---
## 1. Load Data

In [ ]:
import importlib.util, sys as _sys

_joiner_path = repo_root / "scripts" / "join_nba_points_props_actuals_charts_gamelines.py"
_spec = importlib.util.spec_from_file_location("joiner", _joiner_path)
joiner = importlib.util.module_from_spec(_spec)
_sys.modules["joiner"] = joiner
_spec.loader.exec_module(joiner)

# Loads props + game logs + shot charts + game lines fresh from S3 — no stale intermediate
# rim_scorer_pct=40: players with ≥40% of pts from 0-6 ft classified as rim attackers
df_raw = joiner.join_all_data(SEASON, rim_scorer_pct=40)
print(f"\nrows={len(df_raw):,}  cols={df_raw.shape[1]}")
print(f"date range: {df_raw['game_date'].min()} → {df_raw['game_date'].max()}")
df_raw.head(3)

In [ ]:
df = df_raw.copy()

num_cols = ["points_line", "points_over_odds", "points_under_odds", "PTS", "MIN",
            "team_spread", "FGM", "FGA", "FG3M", "FG3A", "FTM", "FTA",
            "OREB", "DREB", "REB", "AST", "STL", "BLK", "TOV",
            "pts_0_6_pct", "rim_fg_pct", "num_bookmakers"]
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["points_line", "PTS"]).copy()

required = ["PLAYER_NAME", "game_date", "MIN", "PTS", "points_line", "team_spread"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print(f"rows after dropna: {len(df):,}")
print(f"players: {df['PLAYER_NAME'].nunique():,}")
print(f"date range: {df['game_date'].min()} → {df['game_date'].max()}")
df.head(3)

---
## 2. EDA

Hypotheses going in:
1. Books are reasonably calibrated on points overall but shade certain player tiers.
2. Spread context (blowout risk) is a systematic pricing variable.
3. Scorer type (rim vs perimeter) adds predictive signal beyond the line itself.

In [ ]:
# Target distribution
print("=== PTS distribution ===")
print(df["PTS"].describe().round(2))

print("\n=== Points line distribution ===")
print(df["points_line"].describe().round(2))

# Over/under/push rates
df["outcome"] = np.where(df["PTS"] > df["points_line"], "over",
                np.where(df["PTS"] < df["points_line"], "under", "push"))
print("\n=== Outcome rates ===")
print(df["outcome"].value_counts(normalize=True).round(3))

In [ ]:
# Market calibration — implied prob vs observed hit rate

def american_to_implied_prob(odds: float) -> float:
    if pd.isna(odds):
        return float("nan")
    return (-odds) / ((-odds) + 100.0) if odds < 0 else 100.0 / (odds + 100.0)

work = df[df["outcome"] != "push"].copy()
work["p_over"] = work["points_over_odds"].apply(american_to_implied_prob)
work["p_under"] = work["points_under_odds"].apply(american_to_implied_prob)
work["y_over"] = (work["outcome"] == "over").astype(int)
work["y_under"] = (work["outcome"] == "under").astype(int)

# Brier score (overall — aggregated odds, not per-book)
brier_over = ((work["p_over"] - work["y_over"]) ** 2).mean()
brier_under = ((work["p_under"] - work["y_under"]) ** 2).mean()
avg_implied_over = work["p_over"].mean()
observed_over = work["y_over"].mean()

print(f"Brier score (over side):  {brier_over:.4f}")
print(f"Brier score (under side): {brier_under:.4f}")
print(f"Avg implied over prob: {avg_implied_over:.3f}  |  Observed over rate: {observed_over:.3f}")
print(f"Calibration gap (over): {abs(avg_implied_over - observed_over):.3f}")

print(f"Avg implied under prob: {work['p_under'].mean():.3f}  |  Observed under rate: {work['y_under'].mean():.3f}")
print(f"Calibration gap (under): {abs(work['p_under'].mean() - work['y_under'].mean()):.3f}")

In [ ]:
# Over rate by points_line tier — does the market shade certain tiers?
df["line_tier"] = pd.cut(
    df["points_line"],
    bins=[0, 10, 15, 20, 25, 30, 100],
    labels=["<10", "10-15", "15-20", "20-25", "25-30", "30+"]
)

tier_stats = (
    df[df["outcome"] != "push"]
    .groupby("line_tier", observed=True)
    .agg(
        n=("PTS", "count"),
        over_rate=("y_over", "mean") if "y_over" in df.columns else ("outcome", lambda x: (x == "over").mean()),
        avg_line=("points_line", "mean"),
        avg_pts=("PTS", "mean"),
    )
    .reset_index()
)

# recompute over_rate cleanly
no_push = df[df["outcome"] != "push"].copy()
no_push["y_over"] = (no_push["outcome"] == "over").astype(int)
tier_stats = (
    no_push.groupby("line_tier", observed=True)
    .agg(n=("PTS", "count"), over_rate=("y_over", "mean"),
         avg_line=("points_line", "mean"), avg_pts=("PTS", "mean"))
    .reset_index()
)
tier_stats.round(3)

In [ ]:
# Over rate by spread bin — blowout risk effect
df["spread_bin"] = pd.cut(
    df["team_spread"],
    bins=[-30, -10, -5, 0, 5, 10, 30],
    labels=["big dog (>10)", "dog (5-10)", "slight dog (0-5)",
            "slight fav (0-5)", "fav (5-10)", "big fav (>10)"]
)
no_push["spread_bin"] = df.loc[no_push.index, "spread_bin"]

spread_stats = (
    no_push.groupby("spread_bin", observed=True)
    .agg(n=("PTS", "count"), over_rate=("y_over", "mean"),
         avg_pts=("PTS", "mean"), avg_line=("points_line", "mean"))
    .reset_index()
)
spread_stats.round(3)

### VERDICT — EDA

- **Calibration**: Market overprices overs by 4% (avg implied 52.5% vs 48.4% actual). Under side tighter (1.7% gap). Vig is ~5.7%. Bet unders at flat odds is directionally correct just from market structure.
- **Line tier**: Over rate decreases monotonically with line height — 48.9% at <10 lines, 43.0% at 30+. Market shades high lines to the under; don't fade this mechanically.
- **Spread**: No meaningful pattern (46–50% across all spread buckets). Blowout risk doesn't move points outcomes beyond what the line already prices in.


---
## 3. Feature Ideation

Key candidate features — test each univariately vs `PTS` and vs `over_hit`:

| Feature | Hypothesis |
|---|---|
| `MIN` rolling avg | Minutes drive opportunity volume |
| `PTS` rolling avg (3/5/10g) | Recent scoring form |
| `team_spread` | Blowout risk / game pace effect |
| `points_line` | Proxy for player role/usage |
| `pts_0_6_pct` | Rim scoring pct — floor consistency signal |
| `scorer_type` | Rim vs perimeter |
| `is_home` | Home court scoring boost |
| `FTA` rolling avg | Free throw rate — trip to line floor |

In [ ]:
# Build rolling features per player (shift(1) = no lookahead)
df_feat = df.sort_values(["PLAYER_NAME", "game_date"]).copy()

for window in [3, 5, 10]:
    df_feat[f"pts_roll{window}"] = (
        df_feat.groupby("PLAYER_NAME")["PTS"]
        .transform(lambda s: s.shift(1).rolling(window, min_periods=window // 2).mean())
    )
    df_feat[f"min_roll{window}"] = (
        df_feat.groupby("PLAYER_NAME")["MIN"]
        .transform(lambda s: s.shift(1).rolling(window, min_periods=window // 2).mean())
    )
    df_feat[f"fta_roll{window}"] = (
        df_feat.groupby("PLAYER_NAME")["FTA"]
        .transform(lambda s: s.shift(1).rolling(window, min_periods=window // 2).mean())
    )

df_feat["y_over"] = (df_feat["outcome"] == "over").astype(int)
df_feat["pts_diff"] = df_feat["PTS"] - df_feat["points_line"]
df_feat["is_home_num"] = df_feat["is_home"].astype(float) if "is_home" in df_feat.columns else np.nan

print(f"rows with pts_roll5: {df_feat['pts_roll5'].notna().sum():,}")
df_feat[["PLAYER_NAME", "game_date", "PTS", "points_line", "y_over",
          "pts_roll5", "min_roll5", "team_spread"]].head(5)

In [ ]:
# Univariate correlations with PTS, y_over, pts_diff
feature_cols = (
    [f"pts_roll{w}" for w in [3, 5, 10]] +
    [f"min_roll{w}" for w in [3, 5, 10]] +
    [f"fta_roll{w}" for w in [3, 5, 10]] +
    ["team_spread", "points_line", "pts_0_6_pct", "is_home_num"]
)
feature_cols = [c for c in feature_cols if c in df_feat.columns]
targets = ["PTS", "y_over", "pts_diff"]

corr = df_feat[feature_cols + targets].corr()[targets].loc[feature_cols]
corr.round(3)

In [ ]:
# Scorer type breakdown — does rim_scorer flag matter?
if "scorer_type" in df_feat.columns:
    scorer_stats = (
        df_feat[df_feat["outcome"] != "push"]
        .groupby("scorer_type")
        .agg(n=("PTS", "count"), over_rate=("y_over", "mean"),
             avg_pts=("PTS", "mean"), avg_line=("points_line", "mean"),
             avg_pts_0_6_pct=("pts_0_6_pct", "mean"))
        .reset_index()
    )
    print(scorer_stats.round(3))

### VERDICT — Features

- **Strongest PTS predictors**: `points_line` (r=0.676) > `pts_roll10` (0.633) > `pts_roll5` (0.614). The line is the prediction.
- **Weakest**: `is_home_num` (r=0.013) — home court irrelevant for points props.
- **Rim attacker signal**: Rim attackers (≥40% pts from 0–6ft) hit unders at 57% vs 51.2% for perimeter players — large gap, small sample (910 vs 14,759). Market may not price scorer type into the line.
- **pts_0_6_pct correlation**: -0.087 with PTS — rim attackers score fewer points vs line. Worth keeping.


---
## 4. Baseline Model

Goal: beat the consensus line RMSE with a simple model.
Naive baseline: predict = line. Compare OLS, Ridge, GBM.

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings("ignore")

FEATURE_COLS_CANDIDATES = [
    "pts_roll5", "min_roll5", "fta_roll5", "team_spread", "points_line",
    "pts_0_6_pct", "is_home_num",
]
FEATURE_COLS = [c for c in FEATURE_COLS_CANDIDATES if c in df_feat.columns]

# Core features required non-NaN; optional features filled with median
CORE_COLS = [c for c in ["pts_roll5", "min_roll5", "team_spread", "points_line"] if c in df_feat.columns]
model_df = df_feat.dropna(subset=CORE_COLS + ["PTS"]).copy()
for col in FEATURE_COLS:
    if col not in CORE_COLS and col in model_df.columns:
        median_val = model_df[col].median()
        model_df[col] = model_df[col].fillna(median_val if pd.notna(median_val) else 0)

# Final safety: drop any rows still NaN in feature matrix
model_df = model_df.dropna(subset=FEATURE_COLS + ["PTS"])

print("Null counts per feature after imputation:")
print(model_df[FEATURE_COLS].isnull().sum())
print(f"Model rows: {len(model_df):,}  features: {FEATURE_COLS}")

X = model_df[FEATURE_COLS].values
y = model_df["PTS"].values

# Naive baseline: predict = line
rmse_naive = np.sqrt(mean_squared_error(y, model_df["points_line"].values))

ols   = LinearRegression()
ridge = Ridge(alpha=1.0)
gbm   = GradientBoostingRegressor(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42)

rmse_ols   = np.sqrt(-cross_val_score(ols,   X, y, cv=5, scoring="neg_mean_squared_error").mean())
rmse_ridge = np.sqrt(-cross_val_score(ridge, X, y, cv=5, scoring="neg_mean_squared_error").mean())
rmse_gbm   = np.sqrt(-cross_val_score(gbm,   X, y, cv=5, scoring="neg_mean_squared_error").mean())

results = pd.DataFrame({
    "model":   ["naive (line)", "OLS", "Ridge", "GBM"],
    "rmse_cv": [rmse_naive, rmse_ols, rmse_ridge, rmse_gbm],
})
results["improvement_vs_naive"] = (rmse_naive - results["rmse_cv"]) / rmse_naive
results.round(4)

In [ ]:
# GBM feature importances
gbm_fit = GradientBoostingRegressor(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42)
gbm_fit.fit(X, y)

pd.Series(gbm_fit.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False).round(4)

### VERDICT — Baseline Model

- **Result**: points_line alone captures 97.97% of GBM feature importance — it IS the prediction. All rolling features add essentially no lift.
- **RMSE**: Naive (line) 6.47 → OLS 6.46 → GBM 6.49. GBM is actually *worse* than OLS, suggesting the small nonlinear signal in rolling stats is noise at this sample size.
- **Takeaway**: Regression-based approach hits a ceiling fast on points. The market is efficiently priced. Edge must come from probability calibration and situational filters, not from beating the line on raw RMSE.


---
## 5. Edge Calibration

Given model prediction vs line, when does a bet have positive EV?
Parametric normal around model mean (same approach as rebounds Option A).

In [ ]:
from scipy.stats import norm

model_df["pred"] = gbm_fit.predict(X)
model_df["resid"] = model_df["PTS"] - model_df["pred"]

sigma = model_df["resid"].std()
print(f"Residual sigma: {sigma:.3f} pts")
print(model_df["resid"].describe().round(3))

In [ ]:
# P(PTS > line) under model distribution
model_df["p_over_model"] = norm.sf(model_df["points_line"], loc=model_df["pred"], scale=sigma)
model_df["p_under_model"] = 1 - model_df["p_over_model"]

model_df["p_over_market"]  = model_df["points_over_odds"].apply(american_to_implied_prob)
model_df["p_under_market"] = model_df["points_under_odds"].apply(american_to_implied_prob)

model_df["edge_over"]  = model_df["p_over_model"]  - model_df["p_over_market"]
model_df["edge_under"] = model_df["p_under_model"] - model_df["p_under_market"]

model_df["edge_bin"] = pd.cut(
    model_df["edge_over"],
    bins=[-1, -0.10, -0.05, 0.0, 0.05, 0.10, 1.0],
    labels=["<-10%", "-10 to -5%", "-5 to 0%", "0 to 5%", "5 to 10%", ">10%"]
)

edge_summary = (
    model_df.dropna(subset=["edge_bin"])
    .groupby("edge_bin", observed=True)
    .agg(
        n=("y_over", "count"),
        actual_over_rate=("y_over", "mean"),
        model_over_prob=("p_over_model", "mean"),
        market_over_prob=("p_over_market", "mean"),
        model_edge=("edge_over", "mean"),
    )
)
edge_summary["roi"] = (edge_summary["actual_over_rate"] / edge_summary["market_over_prob"]) - 1
edge_summary.round(3)


### VERDICT — Edge Calibration (Regression + Normal CDF)

- **Positive ROI threshold**: The 5–10% over edge bin shows ROI ≈ +0.8% (n=297) — marginally positive but tiny.
- **>10% bin anomaly**: ROI 4.4x but avg_p_market = 0.095 — these are near-zero market probability bets, likely data artifacts or extreme outliers. Do not bet these.
- **Under side**: The negative over edge bins (< -5%) represent under edge. Under-edge bins show lower actual over rates, confirming model direction is correct but magnitude is too small to overcome vig.
- **Conclusion**: Regression + Normal CDF approach doesn't generate reliable edge. See Section 6 for direct probability models.


---
## 6. Under-only Direct Probability Models

Train classifiers directly on P(under) — logistic regression and XGBoost.
Two feature sets: rolling pts only (Model A) vs full feature set (Model B).
Evaluate under-only ROI across edge and shrinkage thresholds.

In [ ]:
# ── Target + train/test split ──────────────────────────────────────────────────
# Build clf_df from df_feat so all rolling features are available
ALL_CLF_COLS = ["PLAYER_NAME", "game_date", "PTS", "points_line",
                "points_over_odds", "points_under_odds",
                "pts_roll5", "pts_roll10", "min_roll5", "fta_roll5",
                "team_spread", "pts_0_6_pct", "is_home_num"]

avail = [c for c in ALL_CLF_COLS if c in df_feat.columns]
missing = [c for c in ALL_CLF_COLS if c not in df_feat.columns]
if missing:
    print(f"⚠️  Missing from df_feat: {missing}")

clf_df = df_feat[avail].copy()
clf_df["game_date"] = pd.to_datetime(clf_df["game_date"])
clf_df["points_line"] = pd.to_numeric(clf_df["points_line"], errors="coerce")
clf_df["PTS"] = pd.to_numeric(clf_df["PTS"], errors="coerce")

# Drop rows missing target or minimum features; remove pushes
clf_df = clf_df.dropna(subset=["PTS", "points_line", "pts_roll5"])
clf_df = clf_df[clf_df["PTS"] != clf_df["points_line"]].copy()
clf_df["y_under"] = (clf_df["PTS"] < clf_df["points_line"]).astype(int)

# Chronological 80/20 split
dates_sorted = clf_df["game_date"].sort_values().unique()
cutoff = dates_sorted[int(len(dates_sorted) * 0.80)]
train_clf = clf_df[clf_df["game_date"] < cutoff].copy()
test_clf  = clf_df[clf_df["game_date"] >= cutoff].copy()

print(f"Columns: {avail}")
print(f"Train: {len(train_clf):,} rows  ({train_clf['game_date'].min().date()} → {train_clf['game_date'].max().date()})")
print(f"Test:  {len(test_clf):,} rows  ({test_clf['game_date'].min().date()} → {test_clf['game_date'].max().date()})")
print(f"Under rate — train: {train_clf['y_under'].mean():.3f}  test: {test_clf['y_under'].mean():.3f}")


In [ ]:
# ── Model A vs B: LogReg + XGBoost ────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import brier_score_loss, log_loss
import xgboost as xgb

FEATS_A = ["pts_roll5", "points_line"]
FEATS_B = [f for f in ["pts_roll5", "pts_roll10", "min_roll5", "fta_roll5",
                        "team_spread", "points_line", "pts_0_6_pct", "is_home_num"]
           if f in train_clf.columns]

print(f"Model A features: {FEATS_A}")
print(f"Model B features: {FEATS_B}")

results = []
fitted_models = {}

for feat_name, feats in [("A_rolling_only", FEATS_A), ("B_full", FEATS_B)]:
    train_med = train_clf[feats].median()
    X_tr = train_clf[feats].fillna(train_med)
    X_te = test_clf[feats].fillna(train_med)
    y_tr = train_clf["y_under"]
    y_te = test_clf["y_under"]

    for model_name, model in [
        ("LogReg", Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1000, C=1.0))
        ])),
        ("XGB", xgb.XGBClassifier(
            n_estimators=300, max_depth=3, learning_rate=0.03,
            subsample=0.8, colsample_bytree=0.8,
            eval_metric="logloss", random_state=42, verbosity=0
        )),
    ]:
        model.fit(X_tr, y_tr)
        p_te = model.predict_proba(X_te)[:, 1]
        fitted_models[(feat_name, model_name)] = (model, feats, train_med)
        results.append({
            "features": feat_name,
            "model": model_name,
            "brier": round(brier_score_loss(y_te, p_te), 5),
            "log_loss": round(log_loss(y_te, p_te), 5),
        })

# Market baseline
p_mkt = test_clf["points_under_odds"].apply(american_to_implied_prob)
results.append({
    "features": "market",
    "model": "baseline",
    "brier": round(brier_score_loss(test_clf["y_under"], p_mkt), 5),
    "log_loss": round(log_loss(test_clf["y_under"], p_mkt), 5),
})

res_df = pd.DataFrame(results).sort_values("brier")
print()
print(res_df.to_string(index=False))


In [ ]:
# ── Edge + ROI sweep (under-only, odds-filtered) ─────────────────────────────
# Filter to "main market" props: market P(under) 0.35-0.65 (roughly -190 to +190)
# Excludes extreme long-shots that inflate ROI with single lucky hits.

def american_profit(odds):
    return odds / 100.0 if odds > 0 else 100.0 / abs(odds)

sweep_rows = []

for (feat_name, model_name), (model, feats, train_med) in fitted_models.items():
    X_te   = test_clf[feats].fillna(train_med)
    p_raw  = model.predict_proba(X_te)[:, 1]
    p_mkt  = test_clf["points_under_odds"].apply(american_to_implied_prob).values
    y_act  = test_clf["y_under"].values
    odds_a = test_clf["points_under_odds"].values

    # Odds filter
    odds_ok = (p_mkt >= 0.35) & (p_mkt <= 0.65)

    for shrink in [0.0, 0.25, 0.50]:
        p_adj = p_mkt + (1 - shrink) * (p_raw - p_mkt)
        edge  = p_adj - p_mkt

        for min_edge in [0.01, 0.02, 0.05, 0.08, 0.10]:
            mask  = (edge > min_edge) & odds_ok
            n_bet = int(mask.sum())
            if n_bet == 0:
                continue

            pnl, n_win = 0.0, 0
            for hit, odds in zip(y_act[mask], odds_a[mask]):
                if hit:
                    pnl += american_profit(odds); n_win += 1
                else:
                    pnl -= 1.0

            sweep_rows.append({
                "model": f"{feat_name}/{model_name}",
                "shrink": shrink, "min_edge": min_edge,
                "n_bets": n_bet,
                "hit_rate": round(n_win / n_bet, 3),
                "pnl_u": round(pnl, 1),
                "roi": round(pnl / n_bet, 4),
            })

sweep_df = pd.DataFrame(sweep_rows)

for mdl in sweep_df["model"].unique():
    sub = sweep_df[sweep_df["model"] == mdl]
    pivot   = sub.pivot_table(index="shrink", columns="min_edge", values="roi")
    pivot_n = sub.pivot_table(index="shrink", columns="min_edge", values="n_bets")
    print(f"\n=== ROI  [{mdl}] (market P 0.35-0.65) ===")
    print(pivot.to_string())
    print(f"  n_bets:")
    print(pivot_n.to_string())


In [ ]:
# ── Calibration check (de-vigged) ────────────────────────────────────────────
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

# De-vig market implied probs so they sum to 1 (removes vig bias from calibration plot)
p_ov_raw = test_clf["points_over_odds"].apply(american_to_implied_prob)
p_un_raw = test_clf["points_under_odds"].apply(american_to_implied_prob)
total    = p_ov_raw + p_un_raw
p_mkt_dv = p_un_raw / total   # de-vigged P(under)

fig, ax = plt.subplots(figsize=(7, 5))

for (feat_name, model_name), (model, feats, train_med) in fitted_models.items():
    X_te = test_clf[feats].fillna(train_med)
    p = model.predict_proba(X_te)[:, 1]
    frac_pos, mean_pred = calibration_curve(test_clf["y_under"], p, n_bins=8)
    ax.plot(mean_pred, frac_pos, marker="o", label=f"{feat_name}/{model_name}")

# De-vigged market
frac_pos, mean_pred = calibration_curve(test_clf["y_under"], p_mkt_dv, n_bins=8)
ax.plot(mean_pred, frac_pos, marker="s", linestyle="--", label="market (de-vigged)")

ax.plot([0, 1], [0, 1], "k:", label="perfect")
ax.set_xlim(0.40, 0.65)
ax.set_ylim(0.40, 0.65)
ax.set_xlabel("Mean predicted P(under)")
ax.set_ylabel("Actual under rate")
ax.set_title("Calibration — P(under)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


### VERDICT — Under-only Models

- **Best Brier**: LogReg A_rolling_only (0.24906) — beats market baseline (0.24974). Simpler is better.
- **XGB underperforms market** on Brier (0.24990–0.25126) — overfits the training distribution, doesn't generalize.
- **ROI table is not meaningful here**: test set is playoffs only (Apr–May), structurally different from Oct–Apr training data. N_bets at high thresholds is 15–19 — pure noise.
- **Next step**: Multi-season walk-forward backtest (Section 7) needed for honest ROI estimates.


---
## 7. Multi-season Analysis

Load 2023-24, 2024-25, and 2025-26 together for a proper walk-forward backtest.

**Walk-forward scheme:**
- Fold 1: train on 2023-24 → test on 2024-25
- Fold 2: train on 2023-24 + 2024-25 → test on 2025-26

This gives honest out-of-sample ROI estimates untainted by train/test overlap.


In [ ]:
# ── Load all seasons ─────────────────────────────────────────────────────────
import importlib.util, sys as _sys

_joiner_path = repo_root / "scripts" / "join_nba_points_props_actuals_charts_gamelines.py"
_spec = importlib.util.spec_from_file_location("joiner", _joiner_path)
_joiner = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_joiner)
join_all_data = _joiner.join_all_data

seasons = ["2023-24", "2024-25", "2025-26"]
dfs = {}
for s in seasons:
    print(f"Loading {s}...")
    dfs[s] = join_all_data(s, rim_scorer_pct=40)
    print(f"  {len(dfs[s]):,} rows")

print("\nDone.")


In [ ]:
# ── Build rolling features per season, then concat ───────────────────────────
ALL_CLF_COLS_MS = ["PLAYER_NAME", "game_date", "PTS", "points_line",
                   "points_over_odds", "points_under_odds",
                   "pts_roll5", "pts_roll10", "min_roll5", "fta_roll5",
                   "team_spread", "pts_0_6_pct", "is_home_num", "season"]

season_dfs = []
for s, df_s in dfs.items():
    df_s = df_s.copy()
    num_cols_s = ["points_line", "points_over_odds", "points_under_odds", "PTS", "MIN",
                  "team_spread", "pts_0_6_pct", "is_home_num"]
    for c in num_cols_s:
        if c in df_s.columns:
            df_s[c] = pd.to_numeric(df_s[c], errors="coerce")

    df_s["game_date"] = pd.to_datetime(df_s["GAME_DATE"] if "GAME_DATE" in df_s.columns else df_s["game_date"])
    df_s = df_s.sort_values(["PLAYER_NAME", "game_date"])
    df_s["y_over"] = (df_s["PTS"] > df_s["points_line"]).astype(float)

    for w in [5, 10]:
        df_s[f"pts_roll{w}"] = (df_s.groupby("PLAYER_NAME")["PTS"]
                                    .transform(lambda x: x.shift(1).rolling(w, min_periods=3).mean()))
    for w in [5]:
        df_s[f"min_roll{w}"] = (df_s.groupby("PLAYER_NAME")["MIN"]
                                    .transform(lambda x: x.shift(1).rolling(w, min_periods=3).mean()))
        df_s[f"fta_roll{w}"] = (df_s.groupby("PLAYER_NAME")["FTA"]
                                    .transform(lambda x: x.shift(1).rolling(w, min_periods=3).mean())
                                    if "FTA" in df_s.columns else 0)

    df_s["season"] = s
    avail = [c for c in ALL_CLF_COLS_MS if c in df_s.columns]
    season_dfs.append(df_s[avail])

df_ms = pd.concat(season_dfs, ignore_index=True)
df_ms = df_ms.dropna(subset=["PTS", "points_line", "pts_roll5"])
df_ms = df_ms[df_ms["PTS"] != df_ms["points_line"]].copy()
df_ms["y_under"] = (df_ms["PTS"] < df_ms["points_line"]).astype(int)

print(f"Total rows: {len(df_ms):,}")
print(df_ms.groupby("season")[["y_under", "PTS"]].agg({"y_under": ["count", "mean"], "PTS": "mean"}).round(3))


In [ ]:
# ── Walk-forward backtest ─────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import brier_score_loss
import xgboost as xgb

FEATS_A = ["pts_roll5", "points_line"]
FEATS_B = [f for f in ["pts_roll5", "pts_roll10", "min_roll5", "fta_roll5",
                        "team_spread", "points_line", "pts_0_6_pct", "is_home_num"]
           if f in df_ms.columns]

folds = [
    ("fold1", ["2023-24"],              "2024-25"),
    ("fold2", ["2023-24", "2024-25"],   "2025-26"),
]

wf_results = []
wf_fitted  = {}   # (fold, feat_name, model_name) → (model, feats, train_med)

for fold_name, train_seasons, test_season in folds:
    tr = df_ms[df_ms["season"].isin(train_seasons)]
    te = df_ms[df_ms["season"] == test_season]

    for feat_name, feats in [("A_rolling_only", FEATS_A), ("B_full", FEATS_B)]:
        train_med = tr[feats].median()
        X_tr = tr[feats].fillna(train_med)
        X_te = te[feats].fillna(train_med)
        y_tr = tr["y_under"]
        y_te = te["y_under"]

        for model_name, model in [
            ("LogReg", Pipeline([("sc", StandardScaler()), ("clf", LogisticRegression(max_iter=1000))])),
            ("XGB",    xgb.XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.03,
                                          subsample=0.8, colsample_bytree=0.8,
                                          eval_metric="logloss", random_state=42, verbosity=0)),
        ]:
            model.fit(X_tr, y_tr)
            p_te = model.predict_proba(X_te)[:, 1]
            wf_fitted[(fold_name, feat_name, model_name)] = (model, feats, train_med, te)
            wf_results.append({
                "fold": fold_name,
                "train": str(train_seasons),
                "test": test_season,
                "features": feat_name,
                "model": model_name,
                "brier": round(brier_score_loss(y_te, p_te), 5),
                "n_test": len(y_te),
            })

    # Market baseline per fold
    p_mkt = te["points_under_odds"].apply(american_to_implied_prob)
    wf_results.append({
        "fold": fold_name, "train": str(train_seasons), "test": test_season,
        "features": "market", "model": "baseline",
        "brier": round(brier_score_loss(te["y_under"], p_mkt), 5),
        "n_test": len(te),
    })

wf_df = pd.DataFrame(wf_results)
print(wf_df.to_string(index=False))


In [ ]:
# ── Walk-forward ROI sweep (under-only, odds-filtered) ───────────────────────
# Filter: market P(under) 0.35-0.65 to exclude extreme long-shots

def american_profit(odds):
    return odds / 100.0 if odds > 0 else 100.0 / abs(odds)

wf_roi_rows = []

for (fold_name, feat_name, model_name), (model, feats, train_med, te) in wf_fitted.items():
    X_te   = te[feats].fillna(train_med)
    p_raw  = model.predict_proba(X_te)[:, 1]
    p_mkt  = te["points_under_odds"].apply(american_to_implied_prob).values
    y_act  = te["y_under"].values
    odds_a = te["points_under_odds"].values

    odds_ok = (p_mkt >= 0.35) & (p_mkt <= 0.65)

    for shrink in [0.0, 0.25, 0.50]:
        p_adj = p_mkt + (1 - shrink) * (p_raw - p_mkt)
        edge  = p_adj - p_mkt

        for min_edge in [0.01, 0.02, 0.05, 0.08]:
            mask  = (edge > min_edge) & odds_ok
            n_bet = int(mask.sum())
            if n_bet == 0:
                continue

            pnl, n_win = 0.0, 0
            for hit, odds in zip(y_act[mask], odds_a[mask]):
                if hit:
                    pnl += american_profit(odds); n_win += 1
                else:
                    pnl -= 1.0

            wf_roi_rows.append({
                "fold": fold_name, "test": te["season"].iloc[0],
                "model": f"{feat_name}/{model_name}",
                "shrink": shrink, "min_edge": min_edge,
                "n_bets": n_bet,
                "hit_rate": round(n_win / n_bet, 3),
                "pnl_u": round(pnl, 1),
                "roi": round(pnl / n_bet, 4),
            })

wf_roi = pd.DataFrame(wf_roi_rows)

for fold in wf_roi["fold"].unique():
    print(f"\n{'='*60}")
    print(f"FOLD: {fold}  test={wf_roi[wf_roi['fold']==fold]['test'].iloc[0]}")
    print(f"{'='*60}")
    for mdl in wf_roi["model"].unique():
        sub = wf_roi[(wf_roi["fold"] == fold) & (wf_roi["model"] == mdl)]
        if sub.empty:
            continue
        pivot   = sub.pivot_table(index="shrink", columns="min_edge", values="roi")
        pivot_n = sub.pivot_table(index="shrink", columns="min_edge", values="n_bets")
        print(f"\n  [{mdl}]")
        print(pivot.to_string())
        print(f"  n_bets:")
        print(pivot_n.to_string())


### VERDICT — Multi-season Walk-forward

- **Brier improvement is real**: LogReg consistently beats market baseline in both folds (fold1: 0.25177 vs 0.25307; fold2: 0.24974 vs 0.25225). Simpler features (A) ≈ full features (B). XGB underperforms market in both folds — overfits.
- **ROI is not there** (odds-filtered, market P 0.35–0.65): All models show -5% to -7% ROI in fold1. Fold2 is marginally better (B_full/LogReg +0.57% at min_edge=0.01, n=2,082) but insufficient to beat vig.
- **The calibration edge doesn't translate to profit** at the available sample sizes and feature set.
- **Next directions to explore**:
  1. Situational filters: back-to-backs, pace extremes, matchup defense rating
  2. Rim attacker signal: they hit unders at 57% — test as a standalone filter
  3. Player-level sigma (like rebounds model) instead of global sigma
  4. Extend to line movement as a feature (sharp money signal)


---
## Conclusions — v1 Points Foundations

### What we tried
- Regression (OLS, Ridge, GBM) + parametric Normal CDF edge
- Direct classification (LogReg, XGBoost) on P(under)
- Feature sets: rolling PTS/MIN/FTA, team spread, rim scorer pct, home indicator
- Walk-forward backtest across 3 seasons (2023-24, 2024-25, 2025-26)
- Odds-filtered ROI (market P 0.35–0.65 only)

### What we found

| Finding | Detail |
|---------|--------|
| Market line dominates | 97.97% of GBM feature importance — rolling stats add almost nothing |
| LogReg beats market Brier | Consistently by ~0.001–0.003 across both folds — real but small |
| Not enough to beat vig | After odds filter, fold1 ROI -5% to -7%, fold2 marginally flat. 6.9% vig is a hard ceiling |
| XGBoost overfits | Worse than market on Brier in both folds — too little data for nonlinear models |
| Rim attackers are interesting | 57% under rate vs 51% for perimeter — largest signal found, untested as standalone filter |

### Why points is harder than rebounds

Rebounds has a structural edge: books anchor lines to season averages, ignoring game-to-game variance in minutes and role. Points lines move with the market in real-time — by tip-off, the line already reflects rolling form, injury news, and pace. Almost no information advantage remains.

### What to try next

1. **Rim attacker standalone filter** — bet unders on rim attackers unconditionally, no model needed. 57% hit rate vs 53.5% break-even at -115. Needs walk-forward validation with real odds.
2. **Back-to-back / rest days** — structural fatigue signal not captured in rolling averages.
3. **Game total (over/under)** — low-total games constrain individual scoring; high-total games inflate it. Not in current features.
4. **Line movement** — if the line moves toward the under between open and close, sharp money agrees. Requires tracking opening vs closing lines.
5. **Cross-market signal** — if a player is undervalued on rebounds, does that predict points under? (Correlated markets.)


---
## Scratch

Ad-hoc exploration. Code here does NOT graduate to scripts.